# Gaussian Strong Signal Simulation

Follows the experimental setting in Case 1 of the main text, but with the
$K$-range updated to the current convention: $K_{TEST} = 2,\ldots,\widehat K$
where $\widehat K = \max(25,\,1.75K^*)$ (the reference paper's own
K-range cap, also used as the evaluation scope in the current
`select_r_topK_spearman` method -- see `partA_type2_grid_comparison.py`
and `partB_fixed_datasets_comparison.ipynb`). For $K^*=10$ this gives
$\widehat K=25$ (was $20$ in the original version of this notebook, now
under `archive/`).

**Note:** this notebook has no $r^*$ selection step -- it illustrates the
$r$NN silhouette at 3 FIXED neighborhood sizes ($r \in \{5, 30, 100\}$)
with hand-tuned point-label placement, not a per-replicate selection
algorithm. The hand-tuned label-placement overrides below were tuned for
the OLD range ($K=2..20$); the 5 new points ($K=21..25$) fall back to the
automatic overlap-avoidance logic in `_add_k_labels`, which should still
produce a readable figure, but may not be perfectly decluttered -- check
the rendered figures and add overrides for any new K's that look crowded.
Output figures are saved locally to `figures/paperkmax_buffer10/`
(relative to this notebook's own folder) so they don't overwrite
either the archived version's or the original smoothed-peak-gap
version's figures.


In [ ]:
# -- Core numerical / plotting stack --
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

# -- Statistics: Hungarian algorithm (for clustering accuracy) and Spearman rank correlation --
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr

# -- Clustering (K-means), synthetic Gaussian-mixture data generation, and external validity metrics --
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# -- k-nearest-neighbor queries used to build the rNN affinity distance matrices --
from sklearn.neighbors import NearestNeighbors

## Helper functions

In [ ]:
def simulate_gaussian_data(n_samples=2000, n_features=20, K_star=10,
                           cluster_std=1, random_state=1):
    """Generate mixture-of-Gaussians data (Section: Data Generation).

    Thin wrapper around sklearn's make_blobs: draws `n_samples` points in
    `n_features` dimensions from `K_star` isotropic Gaussian clusters with
    common standard deviation `cluster_std` (cluster centers are placed
    uniformly at random by make_blobs). Returns the unstandardized data
    matrix `X` and the ground-truth cluster labels `y`. `y` is used only
    for external validity metrics (ACC/ARI/NMI below) and is never passed
    to the clustering algorithm itself.
    """
    X, y = make_blobs(n_samples=n_samples, n_features=n_features,
                      centers=K_star, cluster_std=cluster_std,
                      random_state=random_state)
    return X, y


def clustering_accuracy(y_true, y_pred):
    """Clustering accuracy via the Hungarian algorithm.

    Cluster labels produced by K-means are arbitrary integers with no
    inherent correspondence to the true class labels, so "accuracy" first
    requires finding the best 1-to-1 matching between predicted clusters
    and true classes. `w[i, j]` counts how many points fall in predicted
    cluster i AND true class j; `linear_sum_assignment` (Hungarian
    algorithm) finds the permutation that maximizes the total matched
    count (it solves a minimization problem, hence `w.max() - w` as the
    cost matrix). Accuracy is that best matched count divided by the
    total number of points.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    return w[row_ind, col_ind].sum() / y_pred.size


def rnn_affinity_distance(X, r, use_gaussian=False):
    """
    Build rNN affinity distance matrix (precomputed, zeros on diagonal).
    Hard (binary): distance = 0 when either point lists the other among its r nearest neighbors.
    Soft (Gaussian): Gaussian-weighted affinity converted to distance.
    """
    n = X.shape[0]
    # Query r+1 neighbors and remove self so r counts other observations.
    # (Package-wide convention: r = number of NON-SELF neighbors, hence
    # n_neighbors=r+1 followed by dropping column 0 -- sklearn always
    # returns each point itself, at distance 0, as its own first neighbor.)
    nbrs = NearestNeighbors(n_neighbors=r + 1, algorithm='auto').fit(X)
    dists, idx = nbrs.kneighbors(X)
    dists, idx = dists[:, 1:], idx[:, 1:]

    A = np.zeros((n, n))
    if use_gaussian:
        # Soft/Gaussian-weighted affinity: each point i gets its own
        # bandwidth sigma_i = distance to its r-th (farthest of its r)
        # neighbor, so the weighting adapts to local density (denser
        # regions get a smaller bandwidth). Directed (i -> j) affinities
        # are generally asymmetric before the symmetrisation step below.
        for i in range(n):
            sigma = dists[i, -1] or 1e-10   # guard against sigma=0 (coincident points)
            for j_pos, j in enumerate(idx[i]):
                A[i, j] = np.exp(-dists[i, j_pos]**2 / (2 * sigma**2))
    else:
        # Hard/binary affinity: 1 if j is among i's r nearest neighbors, else 0.
        for i in range(n):
            A[i, idx[i]] = 1.0

    A = np.maximum(A, A.T)   # symmetrise (OR rule): i~j if EITHER direction lists the other
    np.fill_diagonal(A, 0)

    # Rescale affinities to [0, 1] before flipping to a distance. For the
    # Hard variant this is a no-op (A is already binary in {0, 1}); for the
    # Soft variant it renormalizes the (asymmetric-bandwidth) Gaussian
    # weights onto a common [0, 1] scale so 1 - A behaves as a proper
    # distance for silhouette_score(metric='precomputed').
    a_min, a_max = A.min(), A.max()
    if a_max > a_min:
        A = (A - a_min) / (a_max - a_min)
    D = 1.0 - A
    np.fill_diagonal(D, 0)
    return D

## Experimental settings

In [ ]:
N           = 2000       # number of samples
d           = 20         # ambient dimensionality
K_STAR      = 10         # true number of Gaussian clusters (ground-truth K*)
CLUSTER_STD = 1          # per-cluster isotropic standard deviation (make_blobs)
RANDOM_STATE = 1         # seed for the representative dataset used throughout this notebook

R_VALUES = [5, 30, 100]          # neighbourhood sizes r (fixed, illustrative -- unrelated to the K-range convention below)

def max_K_eval(k_star):
    '''max(25, 1.75*k*) -- the reference paper's own K-range cap (Section
    3.2), used here as the evaluation scope, matching the current
    partA/partB method.'''
    return int(max(25, round(1.75 * k_star)))

K_MAX  = max_K_eval(K_STAR)
K_TEST = list(range(2, K_MAX + 1))    # K = 2 ... K_MAX -- the K-means sweep evaluated below

print(f'Data : N={N}, d={d}, K_*={K_STAR}, std={CLUSTER_STD}')
print(f'r    : {R_VALUES}')
print(f'K_MAX = max(25, 1.75*K_*) = {K_MAX}')
print(f'K test: {K_TEST}')

## Generate data

In [ ]:
# Generate the single representative dataset used throughout this notebook's
# figures. random_state=1 makes this reproducible, and is the exact seed
# that combined_boxplot_figure.ipynb re-derives (via the same
# simulate_gaussian_data call) when it later needs this identical dataset
# to compute r*, since this notebook itself never computes r*.
X, y_true = simulate_gaussian_data(
    n_samples=N, n_features=d, K_star=K_STAR,
    cluster_std=CLUSTER_STD, random_state=RANDOM_STATE)
print(f'X: {X.shape},  unique labels: {np.unique(y_true)}')

## Pre-compute rNN affinity distance matrices

In [ ]:
# Precompute the rNN affinity distance matrices once per r (both Hard and
# Soft variants). These only depend on X and r, not on the K-means
# clustering, so precomputing them here lets the K-sweep below reuse the
# same matrices across every K instead of rebuilding the r-NN graph
# repeatedly.
dist_hard = {}
dist_soft = {}
for r in R_VALUES:
    print(f'r={r}  hard ...', end=' ')
    dist_hard[r] = rnn_affinity_distance(X, r=r, use_gaussian=False)
    print('soft ...', end=' ')
    dist_soft[r] = rnn_affinity_distance(X, r=r, use_gaussian=True)
    print('done')
print('All distance matrices ready.')

## K-sweep: collect all metrics

In [ ]:
def run_k_sweep(X, y_true, K_test, dist_hard_dict, dist_soft_dict,
                r_values, random_state=42):
    """Cluster X with K-means for every K in K_test and collect, for each K:
    external validity against the ground truth (accuracy, ARI, NMI), the
    ordinary (Euclidean) Silhouette score, and the rNN-based Silhouette
    score (Hard and Soft) at every r in r_values, using the precomputed
    distance matrices passed in.

    Degenerate clusterings (K-means collapsing to a single unique label,
    which can happen for large K when there are duplicate/near-duplicate
    points) are recorded as NaN for every silhouette variant, since
    silhouette_score is undefined when there is only one cluster; NaNs are
    filtered out later wherever Spearman correlations are computed.
    """
    res = dict(K_test=K_test, sil_original=[], accuracy=[], ari=[], nmi=[])
    for r in r_values:
        res[f'sil_hard_r{r}'] = []
        res[f'sil_soft_r{r}'] = []

    for K in K_test:
        labels = KMeans(n_clusters=K, random_state=random_state,
                        n_init='auto').fit_predict(X)
        n_uniq = len(np.unique(labels))

        # External validity metrics only need the predicted labels (not a
        # distance matrix), so they are always computable regardless of
        # whether the clustering degenerated to fewer than K clusters.
        res['accuracy'].append(clustering_accuracy(y_true, labels))
        res['ari'].append(adjusted_rand_score(y_true, labels))
        res['nmi'].append(normalized_mutual_info_score(y_true, labels))

        if n_uniq > 1:
            res['sil_original'].append(
                silhouette_score(X, labels, metric='euclidean'))
            for r in r_values:
                res[f'sil_hard_r{r}'].append(
                    silhouette_score(dist_hard_dict[r], labels,
                                    metric='precomputed'))
                res[f'sil_soft_r{r}'].append(
                    silhouette_score(dist_soft_dict[r], labels,
                                    metric='precomputed'))
        else:
            # Only one cluster came out -- silhouette is undefined here.
            res['sil_original'].append(np.nan)
            for r in r_values:
                res[f'sil_hard_r{r}'].append(np.nan)
                res[f'sil_soft_r{r}'].append(np.nan)

    for key in res:
        if key != 'K_test':
            res[key] = np.array(res[key])
    return res


print('Running K-sweep ...')
results = run_k_sweep(
    X, y_true, K_TEST,
    dist_hard_dict=dist_hard,
    dist_soft_dict=dist_soft,
    r_values=R_VALUES,
)
print('Done.')

In [ ]:
# Cache the K-sweep results to disk. combined_boxplot_figure.ipynb loads
# this exact pickle later (byte-identical reuse) to build the left panel of
# its combined figure, rather than recomputing the K-sweep.
import pickle, os

os.makedirs('data', exist_ok=True)
with open('data/gaussian_simulation_results_paperkmax_buffer10_nonself.pkl', 'wb') as f:
    pickle.dump({'results': results, 'R_VALUES': R_VALUES,
                 'K_TEST': K_TEST, 'K_STAR': K_STAR,
                 'K_MAX': K_MAX,
                 'N': N, 'd': d}, f)
print('Results saved to data/gaussian_simulation_results_paperkmax_buffer10_nonself.pkl')

## Plotting utilities

In [ ]:
import matplotlib.ticker as mticker

def get_point_colors(K_test, K_star):
    """Map each tested K to a color that encodes closeness to true K_star:
    blue shades for K < K_star (under-partition), orange shades for
    K > K_star (over-partition). Color intensity increases the closer K is
    to K_star, so points near the truth are visually more prominent."""
    K_arr     = np.array(K_test)
    dist      = np.abs(K_arr - K_star).astype(float)
    max_d     = dist.max() if dist.max() > 0 else 1.0
    intensity = 1.0 - dist / max_d
    colors = []
    for i, K in enumerate(K_arr):
        v = 0.3 + 0.7 * intensity[i]   # keep colors within a visible range (avoid near-white/near-black extremes)
        colors.append(plt.cm.Blues(v) if K < K_star else plt.cm.Oranges(v))
    return colors, intensity


def _add_k_labels(ax, xs, ys, K_vals,
                  inside_fs=7, outside_fs=7.5,
                  threshold=0.03, offset_frac=0.13,
                  overrides=None):
    """
    Place K number labels with overlap avoidance.
      - Non-overlapping: white bold text centred on the marker.
      - Overlapping: black bold text offset in the repulsion direction,
        connected back to the marker with a thin grey line.

    overrides : dict {k_value -> None | (fx, fy)}
      None     -> force the label inside the marker (white text).
      (fx, fy) -> force the label outside in the given normalised direction.
                  Canonical directions:
                    middle-bottom  (0,   -1)
                    middle-left   (-1,    0)
                    middle-right  ( 1,    0)
                    top            (0,    1)
                    bottom-right  ( 0.707,-0.707)
                    bottom-left  (-0.707,-0.707)
    """
    n = len(xs)
    if n == 0:
        return

    # Normalize point coordinates into [0, 1] axes-fraction space so the
    # collision `threshold` distance is independent of the data's actual
    # scale/units (silhouette scores and external metrics live on very
    # different numeric ranges).
    xlim  = ax.get_xlim()
    ylim  = ax.get_ylim()
    xspan = max(xlim[1] - xlim[0], 1e-9)
    yspan = max(ylim[1] - ylim[0], 1e-9)

    xn = (xs - xlim[0]) / xspan
    yn = (ys - ylim[0]) / yspan

    # Pass 1: flag any point whose label would collide with a nearby
    # point's label (points closer than `threshold` in normalized space).
    needs_offset = set()
    for i in range(n):
        for j in range(i + 1, n):
            if np.hypot(xn[i] - xn[j], yn[i] - yn[j]) < threshold:
                needs_offset.add(i)
                needs_offset.add(j)

    # For each flagged point, compute a "repulsion" direction: sum the
    # unit vectors pointing away from every nearby point (within
    # threshold*5) and normalize. This pushes the offset label away from
    # the local cluster of markers rather than in an arbitrary direction.
    offset_dir = {}
    for i in sorted(needs_offset):
        fx, fy = 0.0, 0.0
        for j in range(n):
            if j == i:
                continue
            dx, dy = xn[i] - xn[j], yn[i] - yn[j]
            d = np.hypot(dx, dy)
            if 0 < d < threshold * 5:
                fx += dx / d
                fy += dy / d
        norm = np.hypot(fx, fy)
        if norm > 1e-6:
            fx, fy = fx / norm, fy / norm
        else:
            # No nearby points to repel away from -- fall back to
            # alternating up/down so co-located labels still separate.
            fx, fy = (0.0, 1.0) if i % 2 == 0 else (0.0, -1.0)
        offset_dir[i] = (fx, fy)

    # Manual overrides take priority over the automatic decision: force a
    # specific K's label inside its marker (None) or outside in a
    # hand-picked direction, for the handful of cases where the automatic
    # repulsion heuristic still produced a crowded/ugly placement.
    if overrides:
        for i in range(n):
            k = int(K_vals[i])
            if k not in overrides:
                continue
            direction = overrides[k]
            if direction is None:
                needs_offset.discard(i)
                offset_dir.pop(i, None)
            else:
                needs_offset.add(i)
                offset_dir[i] = direction

    for i in range(n):
        x, y, k = float(xs[i]), float(ys[i]), int(K_vals[i])
        if i not in needs_offset:
            # No collision (or overridden to "inside"): draw the K number
            # directly on top of the marker in white bold text.
            ax.text(x, y, str(k),
                    ha='center', va='center',
                    fontsize=inside_fs, fontweight='bold',
                    color='white', zorder=6)
        else:
            # Collision: draw the label offset away from the marker, with
            # a thin connecting line back to it (annotate-style callout).
            fx, fy = offset_dir[i]
            tx = x + fx * offset_frac * xspan
            ty = y + fy * offset_frac * yspan
            ax.annotate(
                str(k),
                xy=(x, y), xytext=(tx, ty),
                ha='center', va='center',
                fontsize=outside_fs, fontweight='bold',
                color='black',
                arrowprops=dict(
                    arrowstyle='-', color='gray',
                    lw=0.6, shrinkA=0, shrinkB=4,
                ),
                zorder=6,
            )


def plot_panel(ax, sil, ext, K_test, K_star, r_label, ext_label,
               x_label='Silhouette Score',
               title_fs=11, label_fs=10, tick_fs=9,
               label_overrides=None):
    """Draw one scatter panel: silhouette-type score (x-axis) vs an
    external validity metric (y-axis), one point per tested K.
    Under-partitions (K < K_star) are blue circles, over-partitions
    (K > K_star) are orange squares, and the K achieving the best external
    metric value is marked with a red star. The panel title reports the
    Spearman rank correlation between the x-axis score and the external
    metric -- the central diagnostic this whole simulation is built
    around: higher rho means the internal (silhouette-type) score ranks
    candidate K's more like the external ground-truth metric does."""
    K_arr             = np.array(K_test)
    colors, intensity = get_point_colors(K_test, K_star)
    valid_all         = ~np.isnan(sil) & ~np.isnan(ext)

    for mask, cmap_fn, marker, ms in [
        (K_arr <  K_star, plt.cm.Blues,   'o', 200),
        (K_arr >  K_star, plt.cm.Oranges, 's', 260),  # K=K_star excluded: no square
    ]:
        valid = mask & valid_all
        if not np.any(valid):
            continue
        c_valid = [colors[i] for i in np.where(valid)[0]]
        ax.plot(sil[valid], ext[valid],
                lw=1.5, color=cmap_fn(0.5), alpha=0.3, zorder=1)
        ax.scatter(sil[valid], ext[valid],
                   c=c_valid, s=ms, marker=marker,
                   zorder=3, edgecolors='white', linewidths=0.4)

    # True K* shown only as a red star (no circle/square behind it)
    if valid_all.any():
        best_i = int(np.nanargmax(ext))
        ax.scatter(sil[best_i], ext[best_i],
                   marker='*', s=350, color='red',
                   zorder=5, edgecolors='darkred', linewidths=0.5)

    _add_k_labels(ax, sil[valid_all], ext[valid_all], K_arr[valid_all],
                  overrides=label_overrides)

    rho      = spearmanr(sil[valid_all], ext[valid_all])[0] \
               if valid_all.sum() >= 2 else np.nan
    best_val = np.nanmax(ext)
    best_K   = K_arr[int(np.nanargmax(ext))]

    ax.set_title(
        f'{r_label}\n'
        f'Spearman $\\rho$: {rho:.3f},  '
        f'Best {ext_label}: {best_val:.3f} ($\\mathbf{{K}}$={best_K})',
        fontsize=title_fs, fontweight='bold', pad=4)
    ax.set_xlabel(x_label, fontsize=label_fs)
    ax.set_ylabel(ext_label, fontsize=label_fs)
    ax.tick_params(labelsize=tick_fs)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle=':', alpha=0.4)


def build_legend_handles():
    """Proxy legend entries (best-K star / under-partition circle /
    over-partition square) shared across every panel in a figure."""
    from matplotlib.lines import Line2D
    return [
        Line2D([0], [0], marker='*', color='none', markerfacecolor='red',
               markeredgecolor='darkred', markersize=15,
               label='Best $K$'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='steelblue',
               markeredgecolor='white', markersize=12,
               label='$K <$ True $K^*$  (underestimate)'),
        Line2D([0], [0], marker='s', color='none', markerfacecolor='darkorange',
               markeredgecolor='white', markersize=13,
               label='$K >$ True $K^*$  (overestimate)'),
    ]


def _merge(base, extra):
    """Merge two flat override dicts; extra takes priority."""
    d = dict(base or {})
    d.update(extra or {})
    return d


def _merge_subplot(base_sub, extra_sub):
    """
    Merge two subplot-override dicts {(row,col): {k: dir}}.
    Per-subplot K entries are merged; extra takes priority.
    """
    result = {key: dict(val) for key, val in (base_sub or {}).items()}
    for key, kdict in (extra_sub or {}).items():
        if key in result:
            result[key].update(kdict)
        else:
            result[key] = dict(kdict)
    return result


def make_figure(results, r_values, K_test, K_star,
                ext_key, ext_label, figsize=(16, 13),
                figure_overrides=None,
                figure_row2_overrides=None):
    """
    Assemble the full 3-row comparison figure for one external metric
    (ext_key/ext_label): row 0 is rNN-Hard silhouette panels (one per r in
    r_values), row 1 is rNN-Soft silhouette panels, row 2 is the ordinary
    (Euclidean) silhouette panel plus a shared legend. Each panel is drawn
    by plot_panel() above.

    figure_overrides      : {(row_0idx, col_0idx): {k: dir}} merged on top of
                            the built-in subplot_overrides.
    figure_row2_overrides : {k: dir} merged on top of row2_overrides for the
                            original-silhouette panel (plot (3,1)).
    """
    # -- Font sizes -----------------------------------------------------------
    fs_suptitle  = 14
    fs_row_title = 14
    fs_subtitle  = 11
    fs_axlabel   = 10
    fs_tick      = 10
    fs_legend    = 15
    # -------------------------------------------------------------------------

    # Global: applied to every subplot.
    # K=K_star: no circle/square, so force label inside the red star (white text).
    global_overrides = {
        K_test[-1]: None,   # last tested K always inside its square
        K_star:     None,   # true K* label centred on the red star
    }

    # Built-in per-subplot overrides (shared between both figures)
    base_subplot_overrides = {
        # Row 0 NH ─────────────────────────────────────────────────────────────
        (0, 0): {13: (-1.0,  0.0)},
        (0, 1): {12: None, 13: None, 15: None, 18: (0.0, -1.0)},
        (0, 2): {12: None, 13: None,            18: (0.0, -1.0)},
        # Row 1 NS ─────────────────────────────────────────────────────────────
        (1, 0): {13: (-1.0,  0.0),  9: (0.0,  1.0)},
        (1, 1): {12: None, 14: None, 15: None,  7: (0.0, -1.0)},
        (1, 2): {14: None,                      18: (0.0, -1.0)},
    }

    subplot_overrides = _merge_subplot(base_subplot_overrides, figure_overrides)

    # Row-2 original-silhouette panel
    row2_overrides = _merge(global_overrides, {14: None})
    row2_overrides = _merge(row2_overrides, figure_row2_overrides)

    ext   = results[ext_key]
    ncols = len(r_values)

    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(3, ncols, figure=fig, hspace=0.75, wspace=0.18)

    # --- Row 0: Neighborhood Hard ---
    row0_axes = []
    for col, r in enumerate(r_values):
        ax  = fig.add_subplot(gs[0, col])
        ovr = _merge(global_overrides, subplot_overrides.get((0, col)))
        plot_panel(ax, results[f'sil_hard_r{r}'], ext,
                   K_test, K_star, f'$r = {r}$', ext_label,
                   x_label='Silhouette Score (Neighborhood Hard)',
                   title_fs=fs_subtitle, label_fs=fs_axlabel, tick_fs=fs_tick,
                   label_overrides=ovr)
        row0_axes.append(ax)

    # --- Row 1: Neighborhood Soft ---
    row1_axes = []
    for col, r in enumerate(r_values):
        ax  = fig.add_subplot(gs[1, col])
        ovr = _merge(global_overrides, subplot_overrides.get((1, col)))
        plot_panel(ax, results[f'sil_soft_r{r}'], ext,
                   K_test, K_star, f'$r = {r}$', ext_label,
                   x_label='Silhouette Score (Neighborhood Soft)',
                   title_fs=fs_subtitle, label_fs=fs_axlabel, tick_fs=fs_tick,
                   label_overrides=ovr)
        row1_axes.append(ax)

    # --- Row 2: Original silhouette + legend ---
    ax_orig = fig.add_subplot(gs[2, 0])
    plot_panel(ax_orig, results['sil_original'], ext,
               K_test, K_star, 'Silhouette Score (Original)', ext_label,
               x_label='Silhouette Score (Original)',
               title_fs=fs_subtitle, label_fs=fs_axlabel, tick_fs=fs_tick,
               label_overrides=row2_overrides)

    ax_leg = fig.add_subplot(gs[2, 1:])
    ax_leg.axis('off')
    ax_leg.legend(
        handles=build_legend_handles(),
        loc='center', ncol=1, fontsize=fs_legend, framealpha=0.9,
        handletextpad=0.6, columnspacing=1.5, handlelength=1.0,
    )

    fig.suptitle(
        f'Simulated Gaussian Dataset: Clustering Metrics vs {ext_label}\n'
        f'(True $\\mathbf{{K}} = {K_star}$)',
        fontsize=fs_suptitle, fontweight='bold',
    )
    fig.subplots_adjust(top=0.87)

    for axes_row, row_title in [
        (row0_axes, 'Silhouette Score (Neighborhood Hard)'),
        (row1_axes, 'Silhouette Score (Neighborhood Soft)'),
    ]:
        pos0 = axes_row[0].get_position()
        posN = axes_row[-1].get_position()
        x_center = (pos0.x0 + posN.x1) / 2
        y_top    = pos0.y1
        fig.text(x_center, y_top + 0.03, row_title,
                 ha='center', va='bottom', fontsize=fs_row_title,
                 fontweight='bold', transform=fig.transFigure)

    return fig

## Figure 1 — vs ACC

In [ ]:
# Build the ACC comparison figure. `figure_overrides` below are hand-tuned
# per-panel label-placement overrides for THIS figure specifically (on top
# of the built-in base_subplot_overrides inside make_figure): keys are K
# values, values are None (force label inside its marker) or a unit
# direction vector (force the label outside toward that direction) -- see
# _add_k_labels' docstring above for the canonical direction vocabulary.
# These were manually adjusted by inspecting the rendered figure to fix
# label collisions the automatic repulsion heuristic didn't fully resolve;
# they were tuned for the old K=2..20 range (see the notebook's title cell)
# so double check placement if K_MAX changes.
fig_acc = make_figure(
    results, R_VALUES, K_TEST, K_STAR,
    ext_key='accuracy', ext_label='ACC',
    figure_overrides={
        # (1,1) NH r=5
        (0, 0): { 9: ( 0.0,    1.0),
                 15: ( 0.3,   -0.954)},   # even lower bottom right
        # (1,2) NH r=30
        (0, 1): { 7: ( 0.0,    1.0),
                 11: ( 0.0,    1.0),
                 14: ( 0.0,    1.0),      # top
                 19: ( 0.707, -0.707)},
        # (1,3) NH r=100
        (0, 2): {18: ( 0.707, -0.707),
                 14: ( 0.0,   -1.0),
                 15: ( 0.0,    1.0)},
        # (2,1) NS r=5
        (1, 0): {15: ( 0.3,   -0.954)},  # even lower bottom right
        # (2,2) NS r=30
        (1, 1): {18: (-1.0,    0.0)},
        # (2,3) NS r=100
        (1, 2): {18: ( 0.0,    1.0)},
    },
)
import os
os.makedirs('figures/paperkmax_buffer10', exist_ok=True)
fig_acc.savefig('figures/paperkmax_buffer10/SimGa_vs_ACC.pdf', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — vs ARI

In [ ]:
# Same idea as the ACC figure above, but with hand-tuned label overrides for
# the ARI comparison figure (a different external metric produces a
# different scatter layout, hence a different set of collisions to fix).
fig_ari = make_figure(
    results, R_VALUES, K_TEST, K_STAR,
    ext_key='ari', ext_label='ARI',
    figure_overrides={
        # (1,1) NH r=5
        (0, 0): {14: ( 0.3,   -0.954),   # even lower bottom right
                 15: ( 0.0,   -1.0)},
        # (1,2) NH r=30
        (0, 1): { 7: ( 0.707, -0.707),
                 18: (-1.0,    0.0),
                 19: ( 0.707, -0.707)},
        # (1,3) NH r=100
        (0, 2): {14: None,               # inside the box
                 15: (-0.707,  0.707)},
        # (2,1) NS r=5
        (1, 0): {14: ( 1.0,    0.0),     # right of the box
                 15: ( 0.0,   -1.0)},
        # (2,2) NS r=30
        (1, 1): {13: (-1.0,    0.0),
                  8: ( 0.0,   -1.0),
                 16: None,               # inside the box
                 19: ( 0.0,   -1.0)},
        # (2,3) NS r=100
        (1, 2): {13: None,
                 15: ( 0.0,    1.0),
                 18: (-0.707, -0.707)},
    },
    figure_row2_overrides={13: (0.0, -1.0)},
)
import os
os.makedirs('figures/paperkmax_buffer10', exist_ok=True)
fig_ari.savefig('figures/paperkmax_buffer10/SimGa_vs_ARI.pdf', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — vs NMI

In [ ]:
# NMI comparison figure -- no figure-specific label overrides were needed
# here; the built-in global/subplot overrides inside make_figure() plus the
# automatic repulsion heuristic in _add_k_labels were sufficient for this
# metric's point layout.
fig_nmi = make_figure(
    results, R_VALUES, K_TEST, K_STAR,
    ext_key='nmi', ext_label='NMI',
)
import os
os.makedirs('figures/paperkmax_buffer10', exist_ok=True)
fig_nmi.savefig('figures/paperkmax_buffer10/SimGa_vs_NMI.pdf', dpi=150, bbox_inches='tight')
plt.show()

## Summary table (Spearman correlations)

In [ ]:
# Build a tidy summary table of Spearman rank correlations between each
# score variant (ordinary Silhouette, and rNN-based Hard/Soft Silhouette at
# each fixed r in R_VALUES) and each external validity metric (ACC, ARI,
# NMI). These are the same rho values already printed in each figure's
# panel titles above, collected here into one DataFrame for easy
# side-by-side comparison / export.
import pandas as pd

rows = []
for ext_key, ext_label in [('accuracy', 'ACC'), ('ari', 'ARI'), ('nmi', 'NMI')]:
    ext = results[ext_key]
    sil = results['sil_original']
    m   = ~np.isnan(sil) & ~np.isnan(ext)
    rho = spearmanr(sil[m], ext[m])[0] if m.sum() >= 2 else np.nan
    rows.append({'Method': 'Silhouette Score', 'r': '—',
                 'External': ext_label, 'Spearman rho': round(rho, 4)})
    for r in R_VALUES:
        for variant, lbl in [('hard', 'Neighborhood Hard'),
                              ('soft', 'Neighborhood Soft')]:
            sil = results[f'sil_{variant}_r{r}']
            m   = ~np.isnan(sil) & ~np.isnan(ext)
            rho = spearmanr(sil[m], ext[m])[0] if m.sum() >= 2 else np.nan
            rows.append({'Method': f'Silhouette Score ({lbl})',
                         'r': r, 'External': ext_label,
                         'Spearman rho': round(rho, 4)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))